In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('adult.csv')

print(df.head())

# Remove spaces from text columns
for c in df.select_dtypes(include='object').columns:
    df[c] = df[c].str.strip()

# Replace ? with missing value
df.replace('?', np.nan, inplace=True)

print(df.isnull().sum())

   age workclass  fnlwgt     education  education.num marital.status         occupation   relationship   race     sex  \
0   90         ?   77053       HS-grad              9        Widowed                  ?  Not-in-family  White  Female   
1   82   Private  132870       HS-grad              9        Widowed    Exec-managerial  Not-in-family  White  Female   
2   66         ?  186061  Some-college             10        Widowed                  ?      Unmarried  Black  Female   
3   54   Private  140359       7th-8th              4       Divorced  Machine-op-inspct      Unmarried  White  Female   
4   41   Private  264663  Some-college             10      Separated     Prof-specialty      Own-child  White  Female   

   capital.gain  capital.loss  hours.per.week native.country income  
0             0          4356              40  United-States  <=50K  
1             0          4356              18  United-States  <=50K  
2             0          4356              40  United-States  <

In [9]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({'missing_count': missing_count, 'missing_pct': missing_pct})
missing_report = missing_report[missing_report['missing_count'] > 0].sort_values('missing_count', ascending=False)
print(missing_report)

                missing_count  missing_pct
occupation               1843         5.66
workclass                1836         5.64
native.country            583         1.79


In [13]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

print("Missing values in numeric columns:")
print(df[numeric_cols].isnull().sum())

df_clean = df.copy()


# Fill missing workclass and occupation with "Unknown"
df_clean['workclass'] = df_clean['workclass'].fillna('Unknown')
df_clean['occupation'] = df_clean['occupation'].fillna('Unknown')


# Find the most common country
mode_country = df_clean['native.country'].mode()[0]

print("\nMode of native.country:", mode_country)

# Fill missing native.country with the mode
df_clean['native.country'] = df_clean['native.country'].fillna(mode_country)
#remaining missing values
print("\nMissing values after imputation:")
print(df_clean.isnull().sum())

print("\nTotal missing values remaining:",
      df_clean.isnull().sum().sum())

Missing values in numeric columns:
age               0
fnlwgt            0
education.num     0
capital.gain      0
capital.loss      0
hours.per.week    0
dtype: int64

Mode of native.country: United-States

Missing values after imputation:
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

Total missing values remaining: 0


**Workclass & Occupation:** Replace missing values with "Unknown" because they are categorical and the actual value is unknown.
**Native-country:** Replace missing values with the mode because it is categorical and the most common value is a reasonable replacement.
**Numeric columns:** If missing values exist, use the median because it is less affected by outliers.

In [14]:
exact_dupes = df_clean.duplicated().sum()
print("Exact duplicate rows:", exact_dupes)

cols_no_target = [c for c in df_clean.columns if c != 'income']
dupes_no_target = df_clean.duplicated(subset=cols_no_target).sum()
print("Duplicates ignoring income:", dupes_no_target)

Exact duplicate rows: 24
Duplicates ignoring income: 25


In [19]:
# Check unique values before cleaning
print("Education:")
print(df_clean['education'].unique())

print("\nMarital-status:")
print(df_clean['marital.status'].unique())

print("\nNative-country:")
print(df_clean['native.country'].unique())


# Remove spaces
df_clean['education'] = df_clean['education'].str.strip()
df_clean['marital.status'] = df_clean['marital.status'].str.strip()
df_clean['native.country'] = df_clean['native.country'].str.strip()

# Check unique values after cleaning
print("\n After removing extra spaces")

print("\nEducation:")
print(df_clean['education'].unique())

print("\nMarital-status:")
print(df_clean['marital.status'].unique())

print("\nNative-country:")
print(df_clean['native.country'].unique())

Education:
['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'Preschool']

Marital-status:
['Widowed' 'Divorced' 'Separated' 'Never-married' 'Married-civ-spouse'
 'Married-spouse-absent' 'Married-AF-spouse']

Native-country:
['United-States' 'Mexico' 'Greece' 'Vietnam' 'China' 'Taiwan' 'India'
 'Philippines' 'Trinadad&Tobago' 'Canada' 'South' 'Holand-Netherlands'
 'Puerto-Rico' 'Poland' 'Iran' 'England' 'Germany' 'Italy' 'Japan' 'Hong'
 'Honduras' 'Cuba' 'Ireland' 'Cambodia' 'Peru' 'Nicaragua'
 'Dominican-Republic' 'Haiti' 'El-Salvador' 'Hungary' 'Columbia'
 'Guatemala' 'Jamaica' 'Ecuador' 'France' 'Yugoslavia' 'Scotland'
 'Portugal' 'Laos' 'Thailand' 'Outlying-US(Guam-USVI-etc)']

 After removing extra spaces

Education:
['HS-grad' 'Some-college' '7th-8th' '10th' 'Doctorate' 'Prof-school'
 'Bachelors' 'Masters' '11th' 'Assoc-acdm' 'Assoc-voc' '1st-4th' '5th-6th'
 '12th' '9th' 'P

In [22]:
print("Numeric statistics:")
print(df_clean.describe())

print("\nCategorical statistics:")
print(df_clean.describe(include='object'))

# Mean and median of age
print("\nAge:")
print("Mean:", df_clean['age'].mean())
print("Median:", df_clean['age'].median())

# Mean and median of hours-per-week
print("\nHours per week:")
print("Mean:", df_clean['hours.per.week'].mean())
print("Median:", df_clean['hours.per.week'].median())

Numeric statistics:
                age        fnlwgt  education.num  capital.gain  capital.loss  hours.per.week
count  32561.000000  3.256100e+04   32561.000000  32561.000000  32561.000000    32561.000000
mean      38.581647  1.897784e+05      10.080679   1077.648844     87.303830       40.437456
std       13.640433  1.055500e+05       2.572720   7385.292085    402.960219       12.347429
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000        1.000000
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000       40.000000
50%       37.000000  1.783560e+05      10.000000      0.000000      0.000000       40.000000
75%       48.000000  2.370510e+05      12.000000      0.000000      0.000000       45.000000
max       90.000000  1.484705e+06      16.000000  99999.000000   4356.000000       99.000000

Categorical statistics:
       workclass education      marital.status      occupation relationship   race    sex native.country income
count 

In [24]:
#Most common occupation
print("Most common occupation:")
print(df_clean['occupation'].value_counts().head(1))


# Percentage distribution of sex
print("\nSex percentage distribution:")
print(df_clean['sex'].value_counts(normalize=True) * 100)


# ercentage distribution of income
print("\nIncome percentage distribution:")
print(df_clean['income'].value_counts(normalize=True) * 100)
#IMBALANCED

Most common occupation:
occupation
Prof-specialty    4140
Name: count, dtype: int64

Sex percentage distribution:
sex
Male      66.920549
Female    33.079451
Name: proportion, dtype: float64

Income percentage distribution:
income
<=50K    75.919044
>50K     24.080956
Name: proportion, dtype: float64


In [27]:
# ChecK consistency
education_check = df_clean.groupby('education')['education.num'].unique()

print("Education and education-num mapping:")
print(education_check)

inconsistent = education_check[education_check.apply(len) > 1]

print("\nInconsistencies found:")
print(inconsistent)
#No inconsistencies were found. Each education level consistently maps to a single education-num value.

Education and education-num mapping:
education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education.num, dtype: object

Inconsistencies found:
Series([], Name: education.num, dtype: object)


In [30]:
import pandas as pd
import sqlite3
# Load the original CSV
df_original = pd.read_csv('adult.csv')

#Create a local SQLite database
conn = sqlite3.connect('adult_income.db')
# Load CSV data into a database table
df_original.to_sql('adult_income', conn, if_exists='replace', index=False)

print("CSV data loaded into SQLite table successfully.")
#Run an SQL query
query = """
SELECT *
FROM adult_income
WHERE age > 30;
"""
# Load SQL result into pandas DataFrame
df_sql = pd.read_sql_query(query, conn)

print("\nSQL query result:")
print(df_sql.head())

# Step 6: Compare shapes
print("\nOriginal CSV shape:", df_original.shape)
print("SQL result shape:", df_sql.shape)
conn.close()

CSV data loaded into SQLite table successfully.

SQL query result:
   age workclass  fnlwgt     education  education.num marital.status         occupation   relationship   race     sex  \
0   90         ?   77053       HS-grad              9        Widowed                  ?  Not-in-family  White  Female   
1   82   Private  132870       HS-grad              9        Widowed    Exec-managerial  Not-in-family  White  Female   
2   66         ?  186061  Some-college             10        Widowed                  ?      Unmarried  Black  Female   
3   54   Private  140359       7th-8th              4       Divorced  Machine-op-inspct      Unmarried  White  Female   
4   41   Private  264663  Some-college             10      Separated     Prof-specialty      Own-child  White  Female   

   capital.gain  capital.loss  hours.per.week native.country income  
0             0          4356              40  United-States  <=50K  
1             0          4356              18  United-States  <=50



**Dataset Overview**
The Adult Income dataset has 32,561 rows and 15 columns.
It contains education, employment, and income information.

**Data Quality Issues**
 Missing values were found in workclass, occupation, and native.country.
 Exact duplicates and duplicates ignoring income were checked.
 No inconsistencies were found between education and education.num.

**Key Observations**
 Income classes are imbalanced, with more people earning <=50K.
 The most common occupation was identified using value_counts().
 Mean and median age and working hours were compared.

**Cleaning Decisions**
workclass and occupation missing values → "Unknown".
native.countrymissing values mode
 Extra spaces were removed using .str.strip().
 Exact duplicates were checked, while different income labels were kept.